# Regime Case Screening

This notebook computes simple demand descriptors to help identify candidate benchmark cases across the three datasets.

Target outputs:

- `zero_rate`
- `ADI`
- `CV2`
- regime suggestion

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from src.data_loaders.load_m5 import load_m5_single_series
from src.data_loaders.load_favorita import load_favorita_series
from src.data_loaders.load_amazon import load_amazon_series
from src.models.tsb_model import sbc_classify

In [ ]:
series_map = {
    'm5': load_m5_single_series(base_path='data/raw/m5', random_pick=True, seed=42),
    'favorita': load_favorita_series(base_path='data/raw/favorita', store_nbr=1, family='CLEANING'),
    'amazon': load_amazon_series(
        base_path='data/raw/amazon_2023/review_categories',
        filename='Health_and_Household.jsonl.gz',
        top_rank=1,
        max_rows=100000,
    ),
}

In [ ]:
def compute_case_stats(df: pd.DataFrame, dataset_name: str) -> dict:
    y = pd.to_numeric(df['sales'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
    nz = y[y > 0]
    zero_rate = float((y == 0).mean()) if len(y) else np.nan
    adi = float(len(y) / max((y > 0).sum(), 1)) if len(y) else np.nan
    cv2 = float((np.std(nz) / max(np.mean(nz), 1e-6)) ** 2) if len(nz) > 1 else np.nan
    sbc = sbc_classify(y)
    return {
        'dataset': dataset_name,
        'n_obs': int(len(y)),
        'mean_sales': float(np.mean(y)) if len(y) else np.nan,
        'zero_rate': zero_rate,
        'ADI': adi,
        'CV2': cv2,
        'sbc_regime': sbc['regime'],
        'tsb_domain': sbc['tsb_domain'],
    }

summary = pd.DataFrame([compute_case_stats(df, name) for name, df in series_map.items()])
summary